# KAN Credit Risk Modeling - Phase 2: Baseline Models

This notebook trains, tunes, and evaluates 5 baseline models on the engineered credit risk features: 
1. **Logistic Regression** (baseline linear model)
2. **Random Forest** (baseline bagging ensemble)
3. **XGBoost** (SOTA boosting baseline)
4. **LightGBM** (fast boosting baseline)
5. **Standard MLP** (neural network baseline architecture matching KAN dimensions)

We use **Optuna** for hyperparameter tuning and evaluate all models using **5-Fold Stratified Cross-Validation** to ensure robust performance estimation.

## 1. Environment Configuration, Imports, and Seeds

We configure paths, check for GPU availability, and set random seeds to guarantee reproducibility.

In [ ]:
import os
import gc
import time
import random
import re
import numpy as np
import pandas as pd
import optuna

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss, precision_recall_curve
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# Set seeds for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print("Using device:", DEVICE)

# Optuna logging level
optuna.logging.set_verbosity(optuna.logging.WARNING)

# Locate datasets
DATA_DIR = '/kaggle/input'
train_path = None
test_path = None

# Look for parquet files under Kaggle inputs
for root, dirs, files in os.walk(DATA_DIR):
    for file in files:
        if file == 'train_features.parquet':
            train_path = os.path.join(root, file)
        elif file == 'test_features.parquet':
            test_path = os.path.join(root, file)

# Local fallbacks
if not train_path:
    if os.path.exists('../results/train_features.parquet'):
        train_path = '../results/train_features.parquet'
        test_path = '../results/test_features.parquet'
    elif os.path.exists('./results/train_features.parquet'):
        train_path = './results/train_features.parquet'
        test_path = './results/test_features.parquet'

print("Loaded train path:", train_path)
print("Loaded test path:", test_path)

## 2. Evaluation Utilities

We define a standardized predictions evaluation helper to record uniform performance metrics.

In [ ]:
def evaluate_predictions(y_true, y_pred_proba, train_time=0.0, infer_time=0.0):
    # AUC-ROC
    auc_roc = roc_auc_score(y_true, y_pred_proba)
    
    # PR-AUC
    pr_auc = average_precision_score(y_true, y_pred_proba)
    
    # Optimal F1 and threshold
    precision, recall, thresholds = precision_recall_curve(y_true, y_pred_proba)
    f1_scores = 2 * (precision * recall) / (precision + recall + 1e-10)
    best_idx = np.argmax(f1_scores)
    best_threshold = thresholds[best_idx] if best_idx < len(thresholds) else 0.5
    best_f1 = f1_scores[best_idx]
    
    # Brier Score
    brier = brier_score_loss(y_true, y_pred_proba)
    
    return {
        'auc_roc': auc_roc,
        'pr_auc': pr_auc,
        'f1_score': best_f1,
        'optimal_threshold': best_threshold,
        'brier_score': brier,
        'train_time_sec': train_time,
        'infer_time_sec': infer_time
    }

## 3. Load & Preprocess Data

Since Logistic Regression, Random Forest, and MLP require imputed and scaled data, we apply a robust imputation and scaling pipeline. We keep features formatted as Pandas DataFrames to preserve column name context throughout modeling. We also sanitize feature names to prevent JSON serialization errors in LightGBM.

In [ ]:
# Load datasets
train_df = pd.read_parquet(train_path)
test_df = pd.read_parquet(test_path)

print(f"Train DataFrame Shape: {train_df.shape}")
print(f"Test DataFrame Shape: {test_df.shape}")

# Sanitize column names to retain only alphanumeric characters and underscores,
# which permanently prevents LightGBM JSON character compilation warnings/failures.
def sanitize_column_names(df):
    new_cols = []
    for col in df.columns:
        clean_col = re.sub(r'[^a-zA-Z0-9_]', '_', str(col))
        clean_col = re.sub(r'_+', '_', clean_col).strip('_')
        new_cols.append(clean_col)
    df.columns = new_cols
    return df

train_df = sanitize_column_names(train_df)
test_df = sanitize_column_names(test_df)
print("Feature names sanitized of all non-alphanumeric/special characters.")

# Extract targets and identifiers
y = train_df['TARGET'].values
train_ids = train_df['SK_ID_CURR'].values
test_ids = test_df['SK_ID_CURR'].values

# Drop non-feature columns
features_to_drop = ['TARGET', 'SK_ID_CURR']
feature_cols = [col for col in train_df.columns if col not in features_to_drop]

X = train_df[feature_cols].copy()
X_test = test_df[feature_cols].copy()

# Reduce float precision to save RAM
for col in X.columns:
    if X[col].dtype == np.float64:
        X[col] = X[col].astype(np.float32)
        X_test[col] = X_test[col].astype(np.float32)

print(f"Feature matrix X shape: {X.shape}")

# Imputation & Scaling Pipeline
print("Applying imputation (median) and scaling (standard)... ")
imputer = SimpleImputer(strategy='median')
scaler = StandardScaler()

# Run fitting & convert back to DataFrames to preserve feature names natively
X_imputed = imputer.fit_transform(X)
X_scaled = pd.DataFrame(scaler.fit_transform(X_imputed), columns=feature_cols)

X_test_imputed = imputer.transform(X_test)
X_test_scaled = pd.DataFrame(scaler.transform(X_test_imputed), columns=feature_cols)

print("Data preprocessing completed!")

## 4. PyTorch MLP Implementation

We construct a standard Multi-Layer Perceptron (MLP) architecture in PyTorch as a direct baseline comparison for Kolmogorov-Arnold Networks (KAN).

In [ ]:
class SimpleMLP(nn.Module):
    def __init__(self, input_dim, hidden_dims=[128, 64], dropout=0.2):
        super(SimpleMLP, self).__init__()
        layers = []
        in_dim = input_dim
        for h_dim in hidden_dims:
            layers.append(nn.Linear(in_dim, h_dim))
            layers.append(nn.BatchNorm1d(h_dim))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout))
            in_dim = h_dim
        layers.append(nn.Linear(in_dim, 1))
        self.network = nn.Sequential(*layers)
        
    def forward(self, x):
        return self.network(x)

def train_mlp_baseline(X_train, y_train, X_val, y_val, hidden_dims=[128, 64], 
                       lr=1e-3, batch_size=512, epochs=30, dropout=0.2, 
                       weight_decay=1e-5, device='cpu'):
    # Convert to Tensors (extracting underlying numpy arrays from DataFrames if necessary)
    X_train_np = X_train.values if isinstance(X_train, pd.DataFrame) else X_train
    X_val_np = X_val.values if isinstance(X_val, pd.DataFrame) else X_val
    
    X_train_t = torch.tensor(X_train_np, dtype=torch.float32)
    y_train_t = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)
    X_val_t = torch.tensor(X_val_np, dtype=torch.float32)
    y_val_t = torch.tensor(y_val, dtype=torch.float32).unsqueeze(1)
    
    train_dataset = TensorDataset(X_train_t, y_train_t)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    
    model = SimpleMLP(X_train_np.shape[1], hidden_dims, dropout).to(device)
    
    # Class weights logic
    neg_count = (y_train == 0).sum()
    pos_count = (y_train == 1).sum()
    pos_weight = torch.tensor([neg_count / (pos_count + 1e-5)], dtype=torch.float32).to(device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    
    best_val_loss = float('inf')
    best_model_state = None
    patience = 5
    patience_counter = 0
    
    start_time = time.time()
    for epoch in range(epochs):
        model.train()
        for batch_X, batch_y in train_loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            optimizer.zero_grad()
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()
            
        # Validate
        model.eval()
        with torch.no_grad():
            val_X_device = X_val_t.to(device)
            val_y_device = y_val_t.to(device)
            val_outputs = model(val_X_device)
            val_loss = criterion(val_outputs, val_y_device).item()
            
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            # Fixed PyTorch bug: deep clone state dict to CPU so reference updates during training are decoupled
            best_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                break
                
    train_time = time.time() - start_time
    
    if best_model_state is not None:
        model.load_state_dict(best_model_state)
        
    model.eval()
    with torch.no_grad():
        start_infer = time.time()
        val_outputs = torch.sigmoid(model(X_val_t.to(device))).cpu().numpy().flatten()
        infer_time = time.time() - start_infer
        
    return val_outputs, train_time, infer_time

## 5. Hyperparameter Tuning & Cross-Validation Pipeline

For each of our 5 models, we create an Optuna study to optimize hyperparameters on validation splits, followed by full 5-fold training.

In [ ]:
# Dictionary to store performance results
results = {}

# Standard 5-fold split used uniformly across all models
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
fold_indices = list(skf.split(X_scaled, y))

print("Validation set splits created. Ready for training.")

### 5.1 Logistic Regression

In [ ]:
print("--- Hyperparameter Tuning: Logistic Regression ---")

def lr_objective(trial):
    C = trial.suggest_float('C', 1e-4, 10.0, log=True)
    
    # Fast evaluation on Fold 0
    train_idx, val_idx = fold_indices[0]
    X_train, y_train = X_scaled.iloc[train_idx], y[train_idx]
    X_val, y_val = X_scaled.iloc[val_idx], y[val_idx]
    
    # Increased max_iter to 1000 to solve convergence warnings
    model = LogisticRegression(C=C, max_iter=1000, random_state=SEED, class_weight='balanced', n_jobs=-1)
    model.fit(X_train, y_train)
    preds = model.predict_proba(X_val)[:, 1]
    return roc_auc_score(y_val, preds)

study_lr = optuna.create_study(direction='maximize')
study_lr.optimize(lr_objective, n_trials=10)
best_C = study_lr.best_params['C']
print(f"Best Params: C={best_C:.5f}")

# 5-Fold Training
oof_lr = np.zeros(len(y))
total_train_time = 0
total_infer_time = 0

for fold, (train_idx, val_idx) in enumerate(fold_indices):
    X_train, y_train = X_scaled.iloc[train_idx], y[train_idx]
    X_val, y_val = X_scaled.iloc[val_idx], y[val_idx]
    
    # Increased max_iter to 1000 to solve convergence warnings
    model = LogisticRegression(C=best_C, max_iter=1000, random_state=SEED, class_weight='balanced', n_jobs=-1)
    
    start_t = time.time()
    model.fit(X_train, y_train)
    total_train_time += (time.time() - start_t)
    
    start_i = time.time()
    oof_lr[val_idx] = model.predict_proba(X_val)[:, 1]
    total_infer_time += (time.time() - start_i)

results['Logistic Regression'] = evaluate_predictions(y, oof_lr, total_train_time, total_infer_time)
print("Logistic Regression Evaluation:", results['Logistic Regression'])

### 5.2 Random Forest

In [ ]:
print("--- Hyperparameter Tuning: Random Forest ---")

def rf_objective(trial):
    n_estimators = trial.suggest_int('n_estimators', 50, 150)
    max_depth = trial.suggest_int('max_depth', 5, 12)
    
    train_idx, val_idx = fold_indices[0]
    X_train, y_train = X_scaled.iloc[train_idx], y[train_idx]
    X_val, y_val = X_scaled.iloc[val_idx], y[val_idx]
    
    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        random_state=SEED,
        class_weight='balanced',
        n_jobs=-1
    )
    model.fit(X_train, y_train)
    preds = model.predict_proba(X_val)[:, 1]
    return roc_auc_score(y_val, preds)

study_rf = optuna.create_study(direction='maximize')
study_rf.optimize(rf_objective, n_trials=8)
best_rf = study_rf.best_params
print(f"Best Params: {best_rf}")

# 5-Fold Training
oof_rf = np.zeros(len(y))
total_train_time = 0
total_infer_time = 0

for fold, (train_idx, val_idx) in enumerate(fold_indices):
    X_train, y_train = X_scaled.iloc[train_idx], y[train_idx]
    X_val, y_val = X_scaled.iloc[val_idx], y[val_idx]
    
    model = RandomForestClassifier(
        n_estimators=best_rf['n_estimators'],
        max_depth=best_rf['max_depth'],
        random_state=SEED,
        class_weight='balanced',
        n_jobs=-1
    )
    
    start_t = time.time()
    model.fit(X_train, y_train)
    total_train_time += (time.time() - start_t)
    
    start_i = time.time()
    oof_rf[val_idx] = model.predict_proba(X_val)[:, 1]
    total_infer_time += (time.time() - start_i)

results['Random Forest'] = evaluate_predictions(y, oof_rf, total_train_time, total_infer_time)
print("Random Forest Evaluation:", results['Random Forest'])

### 5.3 XGBoost

In [ ]:
print("--- Hyperparameter Tuning: XGBoost ---")

# Class balance scaling weight
scale_pos_weight = (len(y) - y.sum()) / y.sum()

def xgb_objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 150),
        'max_depth': trial.suggest_int('max_depth', 3, 7),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0)
    }
    
    train_idx, val_idx = fold_indices[0]
    X_train, y_train = X_scaled.iloc[train_idx], y[train_idx]
    X_val, y_val = X_scaled.iloc[val_idx], y[val_idx]
    
    # Changed device to 'cpu' for XGBoost evaluation to avoid device mismatch warnings
    # between GPU booster memory and CPU validation arrays, since CPU evaluation is highly efficient.
    model = XGBClassifier(
        **params,
        scale_pos_weight=scale_pos_weight,
        random_state=SEED,
        n_jobs=-1,
        tree_method='hist',
        device='cpu'
    )
    model.fit(X_train, y_train)
    preds = model.predict_proba(X_val)[:, 1]
    return roc_auc_score(y_val, preds)

study_xgb = optuna.create_study(direction='maximize')
study_xgb.optimize(xgb_objective, n_trials=10)
best_xgb = study_xgb.best_params
print(f"Best Params: {best_xgb}")

# 5-Fold Training
oof_xgb = np.zeros(len(y))
total_train_time = 0
total_infer_time = 0

for fold, (train_idx, val_idx) in enumerate(fold_indices):
    X_train, y_train = X_scaled.iloc[train_idx], y[train_idx]
    X_val, y_val = X_scaled.iloc[val_idx], y[val_idx]
    
    model = XGBClassifier(
        **best_xgb,
        scale_pos_weight=scale_pos_weight,
        random_state=SEED,
        n_jobs=-1,
        tree_method='hist',
        device='cpu'
    )
    
    start_t = time.time()
    model.fit(X_train, y_train)
    total_train_time += (time.time() - start_t)
    
    start_i = time.time()
    oof_xgb[val_idx] = model.predict_proba(X_val)[:, 1]
    total_infer_time += (time.time() - start_i)

results['XGBoost'] = evaluate_predictions(y, oof_xgb, total_train_time, total_infer_time)
print("XGBoost Evaluation:", results['XGBoost'])

### 5.4 LightGBM

In [ ]:
print("--- Hyperparameter Tuning: LightGBM ---")

def lgbm_objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 150),
        'max_depth': trial.suggest_int('max_depth', 3, 8),
        'num_leaves': trial.suggest_int('num_leaves', 15, 63),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0)
    }
    
    train_idx, val_idx = fold_indices[0]
    # Slicing the Pandas DataFrames keeps columns & feature names matched, resolving the UserWarning permanently
    X_train, y_train = X_scaled.iloc[train_idx], y[train_idx]
    X_val, y_val = X_scaled.iloc[val_idx], y[val_idx]
    
    model = LGBMClassifier(
        **params,
        class_weight='balanced',
        random_state=SEED,
        n_jobs=-1,
        verbose=-1
    )
    model.fit(X_train, y_train)
    preds = model.predict_proba(X_val)[:, 1]
    return roc_auc_score(y_val, preds)

study_lgbm = optuna.create_study(direction='maximize')
study_lgbm.optimize(lgbm_objective, n_trials=10)
best_lgbm = study_lgbm.best_params
print(f"Best Params: {best_lgbm}")

# 5-Fold Training
oof_lgbm = np.zeros(len(y))
total_train_time = 0
total_infer_time = 0

for fold, (train_idx, val_idx) in enumerate(fold_indices):
    X_train, y_train = X_scaled.iloc[train_idx], y[train_idx]
    X_val, y_val = X_scaled.iloc[val_idx], y[val_idx]
    
    model = LGBMClassifier(
        **best_lgbm,
        class_weight='balanced',
        random_state=SEED,
        n_jobs=-1,
        verbose=-1
    )
    
    start_t = time.time()
    model.fit(X_train, y_train)
    total_train_time += (time.time() - start_t)
    
    start_i = time.time()
    oof_lgbm[val_idx] = model.predict_proba(X_val)[:, 1]
    total_infer_time += (time.time() - start_i)

results['LightGBM'] = evaluate_predictions(y, oof_lgbm, total_train_time, total_infer_time)
print("LightGBM Evaluation:", results['LightGBM'])

### 5.5 PyTorch MLP

In [ ]:
print("--- Hyperparameter Tuning: PyTorch MLP ---")

def mlp_objective(trial):
    lr = trial.suggest_float('lr', 1e-4, 1e-2, log=True)
    dropout = trial.suggest_float('dropout', 0.1, 0.4)
    h_dim = trial.suggest_categorical('h_dim', [64, 128])
    
    train_idx, val_idx = fold_indices[0]
    X_train, y_train = X_scaled.iloc[train_idx], y[train_idx]
    X_val, y_val = X_scaled.iloc[val_idx], y[val_idx]
    
    preds, _, _ = train_mlp_baseline(
        X_train, y_train, X_val, y_val,
        hidden_dims=[h_dim, h_dim // 2],
        lr=lr,
        dropout=dropout,
        epochs=10,
        device=DEVICE
    )
    return roc_auc_score(y_val, preds)

study_mlp = optuna.create_study(direction='maximize')
study_mlp.optimize(mlp_objective, n_trials=6)
best_mlp = study_mlp.best_params
print(f"Best Params: {best_mlp}")

# 5-Fold Training
oof_mlp = np.zeros(len(y))
total_train_time = 0
total_infer_time = 0

for fold, (train_idx, val_idx) in enumerate(fold_indices):
    X_train, y_train = X_scaled.iloc[train_idx], y[train_idx]
    X_val, y_val = X_scaled.iloc[val_idx], y[val_idx]
    
    preds, t_time, i_time = train_mlp_baseline(
        X_train, y_train, X_val, y_val,
        hidden_dims=[best_mlp['h_dim'], best_mlp['h_dim'] // 2],
        lr=best_mlp['lr'],
        dropout=best_mlp['dropout'],
        epochs=30,
        device=DEVICE
    )
    
    oof_mlp[val_idx] = preds
    total_train_time += t_time
    total_infer_time += i_time

results['Standard MLP'] = evaluate_predictions(y, oof_mlp, total_train_time, total_infer_time)
print("Standard MLP Evaluation:", results['Standard MLP'])

# Clean GPU memory caches after deep learning models to conserve RAM on Kaggle Free Tier
torch.cuda.empty_cache()
gc.collect()

## 6. Summary Comparison Table & Saving Metrics

We aggregate the results of all 5 baselines, print a visual comparison, and save/append them to `metrics_summary.csv`.

In [ ]:
# Construct Summary DataFrame
metrics_df = pd.DataFrame(results).T
metrics_df.index.name = 'Model'
metrics_df = metrics_df.reset_index()

column_mapping = {
    'auc_roc': 'AUC-ROC',
    'pr_auc': 'PR-AUC',
    'f1_score': 'F1-Score',
    'brier_score': 'Brier Score',
    'train_time_sec': 'Training Time',
    'infer_time_sec': 'Inference Time'
}
metrics_df = metrics_df.rename(columns=column_mapping)
metrics_df = metrics_df[['Model', 'AUC-ROC', 'PR-AUC', 'F1-Score', 'Brier Score', 'Training Time', 'Inference Time']]

print("\n--- Model Performance Comparison ---")
print(metrics_df.to_markdown(index=False))

output_csv = '/kaggle/working/metrics_summary.csv'
if not os.path.exists('/kaggle/working'):
    output_csv = './results/metrics_summary.csv'
    
metrics_df.to_csv(output_csv, index=False)
print(f"\nMetrics successfully written to: {output_csv}")